# General Analysis of the Arthouse Cohort
## From Analytics to Action — DTU Spring 2026

**Working set:** the 4,191-film arthouse cohort from `arthouse_cohort.csv` — the union of the rule-based filter and the LLM scorer at `arthouse_score >= 8`. See `deciding-method.md` for the cohort definition and `coverage.ipynb` for field-level coverage.

This notebook walks **six sections** of focused KPIs. Every metric was chosen because it answers a real question for Publikum's four decision questions: positioning, target audiences, market strategy, comparable titles. Where coverage is too thin, we say so explicitly rather than ship a misleading number.

## §1 — Cohort overview & data quality

Anchor numbers. Without these, every later percentage floats.

In [ ]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Find project root
PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / '03-data').exists() and (candidate / 'src').exists():
        PROJECT_ROOT = candidate
        break
os.chdir(PROJECT_ROOT)

cohort = pd.read_csv('notebooks/arthouse/arthouse-analysis/arthouse_cohort.csv')
universe = pd.read_csv('notebooks/arthouse/arthouse-LLM-classification/films_arthouse_scored.csv')
non_cohort = universe.loc[~universe['titleId'].isin(cohort['titleId'])].copy()

# Plot palette — kept consistent across the whole notebook
PRIMARY = '#3a5a78'   # cohort / headline
ACCENT  = '#c46d4a'   # secondary highlight
GRAY    = '#888888'   # baseline / non-cohort

print(f'Universe (all scored films): {len(universe):,}')
print(f'Arthouse cohort:             {len(cohort):,}  ({len(cohort)/len(universe)*100:.1f}%)')
print(f'Non-cohort (baseline):       {len(non_cohort):,}')
print(f'Year range:                  {int(cohort["releaseYear"].min())} – {int(cohort["releaseYear"].max())}, median {int(cohort["releaseYear"].median())}')

In [ ]:
# Source split — rule-only / llm-only / both
src = cohort['arthouse_source'].value_counts().rename_axis('source').to_frame('films')
src['pct'] = (src['films'] / len(cohort) * 100).round(1)
display(Markdown('**Cohort source split**'))
display(src)

# Coverage of the five fields this notebook actually depends on.
# Rating distribution requires totalVotes >= 100 to be statistically usable.
fields = {
    'IMDb rating':              cohort['imdbRating'].notna(),
    'Rating distribution (>=100 votes)': cohort['rating_1'].notna() & (cohort['totalVotes'].fillna(0) >= 100),
    'Keywords':                 cohort['keywords'].notna(),
    'mainCountry':              cohort['mainCountry'].notna(),
    'releaseYear':              cohort['releaseYear'].notna(),
    'Budget AND revenue (>0)':  (cohort['budget'].fillna(0) > 0) & (cohort['revenue'].fillna(0) > 0),
}
cov = pd.DataFrame({
    'films': [int(m.sum()) for m in fields.values()],
    'pct':   [round(m.mean()*100, 1) for m in fields.values()],
}, index=list(fields.keys())).rename_axis('field used in analysis')
display(Markdown('**Coverage of fields used here**'))
display(cov)

**Reading the table:** the four fields driving §2–§6 all sit at 60%+. The commercial fields used in §7 sit at ~5% — every chart there will carry an explicit n caveat.

## §2 — Audience reception

The most novel section. Two questions: (1) is arthouse received differently from the rest, and (2) does it polarize audiences more?

**Polarization score** = share of votes in extreme tails:
$$\text{polar} = \frac{r_1 + r_2 + r_9 + r_{10}}{\text{totalVotes}}$$

A unimodal "everyone agrees this is fine" film sits low (most votes in 5–8). A bimodal "love-it-or-hate-it" film sits high. Filter: ≥100 votes — fewer than that and the distribution is statistical noise.

In [ ]:
def polarization(df):
    """Return frame of films with usable rating distribution and a 'polar' column."""
    mask = df['rating_1'].notna() & (df['totalVotes'].fillna(0) >= 100)
    sub = df.loc[mask].copy()
    tail = sub['rating_1'] + sub['rating_2'] + sub['rating_9'] + sub['rating_10']
    sub['polar'] = tail / sub['totalVotes']
    weighted = sum(i * sub[f'rating_{i}'] for i in range(1, 11))
    sub['mean_from_dist'] = weighted / sub['totalVotes']
    return sub

coh_pol = polarization(cohort)
non_pol = polarization(non_cohort)

stats = pd.DataFrame({
    'Cohort':     [len(coh_pol),
                   round(coh_pol['imdbRating'].mean(), 2),
                   round(coh_pol['polar'].mean(), 3),
                   round((coh_pol['imdbRating'] >= 7.5).mean() * 100, 1)],
    'Non-cohort': [len(non_pol),
                   round(non_pol['imdbRating'].mean(), 2),
                   round(non_pol['polar'].mean(), 3),
                   round((non_pol['imdbRating'] >= 7.5).mean() * 100, 1)],
}, index=['n (>=100 votes)', 'Mean IMDb rating', 'Mean polarization', '% rated >= 7.5'])
display(Markdown('**Cohort vs non-cohort, films with >=100 votes**'))
display(stats)

# Headline gap
gap_rating = coh_pol['imdbRating'].mean() - non_pol['imdbRating'].mean()
gap_polar  = coh_pol['polar'].mean() - non_pol['polar'].mean()
print(f'Rating gap:        {gap_rating:+.2f}  (cohort minus non-cohort)')
print(f'Polarization gap:  {gap_polar:+.3f}')

In [ ]:
# Distribution of polarization scores — overlapping histograms
fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(0, 1, 41)
ax.hist(non_pol['polar'], bins=bins, alpha=0.55, color=GRAY,
        label=f'Non-cohort (n={len(non_pol):,})', density=True)
ax.hist(coh_pol['polar'], bins=bins, alpha=0.7, color=PRIMARY,
        label=f'Cohort (n={len(coh_pol):,})', density=True)
ax.axvline(coh_pol['polar'].mean(), color=PRIMARY, ls='--', lw=1)
ax.axvline(non_pol['polar'].mean(), color=GRAY, ls='--', lw=1)
ax.set_xlabel('Polarization score = share of votes in 1–2 + 9–10')
ax.set_ylabel('Density')
ax.set_title('Polarization distribution — arthouse vs everything else')
ax.legend(frameon=False)
for s in ('top','right'): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# Three illustrative cohort films, picked to show different distribution shapes.
# Use >=1000 votes for clean visual shape.
big = coh_pol[coh_pol['totalVotes'] >= 1000].copy()

most_polar = big.nlargest(1, 'polar').iloc[0]
consensus  = big[big['polar'] < big['polar'].median()].nlargest(1, 'imdbRating').iloc[0]
disliked   = big.nsmallest(1, 'mean_from_dist').iloc[0]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, row, lbl in zip(axes,
                        [most_polar, consensus, disliked],
                        ['Most polarized', 'Critical consensus', 'Lowest-rated']):
    counts = [row[f'rating_{i}'] for i in range(1, 11)]
    ax.bar(range(1, 11), counts, color=PRIMARY)
    title = row.get('englishTitle') if pd.notna(row.get('englishTitle')) else row.get('originalTitle')
    title = str(title)[:38] if title else '?'
    ax.set_title(f"{lbl}\n{title} ({int(row['releaseYear'])})\nmean={row['mean_from_dist']:.1f}, polar={row['polar']:.2f}",
                 fontsize=9)
    ax.set_xticks(range(1, 11))
    ax.set_xlabel('rating')
    for s in ('top','right'): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

**Implication for Publikum (positioning + risk):** the polarization score is a per-film number — it travels into the comparable-title workflow. A high-polar comp predicts **review variance**, which matters for marketing tone and for setting expectations with stakeholders.

## §3 — Geography & market

Where arthouse comes from, and which markets punch above their weight.

In [ ]:
# 2020 population estimates (millions) for European + a few non-European producing countries.
# Source: UN/Worldometer; precision is fine for the per-capita ranking.
POPULATION_M = {
    'FR': 67.4, 'DE': 83.2, 'IT': 59.6, 'GB': 67.0, 'ES': 47.4,
    'BE': 11.5, 'PT': 10.3, 'AT': 9.0,  'CH': 8.6,  'HU': 9.7,
    'NL': 17.4, 'SE': 10.4, 'PL': 38.0, 'DK': 5.8,  'GR': 10.7,
    'NO': 5.4,  'CZ': 10.7, 'IE': 5.0,  'FI': 5.5,  'RO': 19.3,
    'IS': 0.37, 'LU': 0.63, 'EE': 1.3,  'LV': 1.9,  'LT': 2.8,
    'BG': 6.9,  'HR': 4.1,  'RS': 6.9,  'SK': 5.5,  'SI': 2.1,
    'US': 331.0, 'CA': 38.0, 'JP': 125.8, 'KR': 51.8,
}

cc = cohort['mainCountry'].value_counts()

top15 = cc.head(15).to_frame('films')
top15['% of cohort'] = (top15['films'] / len(cohort) * 100).round(1)
top15['per million'] = [round(top15.loc[c, 'films'] / POPULATION_M[c], 1)
                         if c in POPULATION_M else np.nan for c in top15.index]
display(Markdown('**Top 15 producing countries**'))
display(top15)

# Per-capita ranking, restricted to countries with >=15 films and known population
elig = cc[cc >= 15]
per_cap_rows = []
for c, n in elig.items():
    if c in POPULATION_M:
        per_cap_rows.append({'country': c, 'films': int(n),
                             'per million': round(n / POPULATION_M[c], 1)})
per_cap = pd.DataFrame(per_cap_rows).sort_values('per million', ascending=False).head(10).set_index('country')
display(Markdown('**Top 10 per-capita arthouse output (>=15 films)**'))
display(per_cap)

In [ ]:
# Language split
lang = cohort['original_language'].fillna(cohort['firstLanguage'])
known = lang.notna().sum()
en = (lang == 'en').sum()
print(f'English-language cohort films:    {en:>5,} of {known:,} known ({en/known*100:.1f}%)')
print(f'Most common non-English languages: ' + ', '.join(
    f"{l} ({n:,})" for l, n in lang[lang != 'en'].value_counts().head(5).items()
))

# Country x decade heatmap for the top 5 countries
top5 = cc.head(5).index.tolist()
sub = cohort[cohort['mainCountry'].isin(top5) & cohort['releaseYear'].notna()].copy()
sub['decade'] = (sub['releaseYear'] // 10 * 10).astype(int)
sub = sub[(sub['decade'] >= 1960) & (sub['decade'] <= 2020)]
heat = sub.groupby(['decade', 'mainCountry']).size().unstack(fill_value=0).reindex(columns=top5)

fig, ax = plt.subplots(figsize=(10, 3.2))
im = ax.imshow(heat.T.values, aspect='auto', cmap='Blues')
ax.set_yticks(range(len(top5))); ax.set_yticklabels(top5)
ax.set_xticks(range(len(heat.index))); ax.set_xticklabels([f'{int(d)}s' for d in heat.index])
ax.set_title('Cohort films per decade — top 5 producing countries')
vmax = heat.values.max()
for i in range(len(top5)):
    for j in range(len(heat.index)):
        v = heat.T.values[i, j]
        ax.text(j, i, str(v), ha='center', va='center', fontsize=8,
                color='white' if v > vmax * 0.5 else 'black')
fig.colorbar(im, ax=ax, shrink=0.7, label='films')
plt.tight_layout(); plt.show()

**Implication for Publikum (market strategy):** the per-capita ranking is the strategic lens — France produces a *lot* in absolute terms, but several smaller countries punch much harder per capita and may be better comp markets for similar-scale films. The heatmap shows pipeline shifts decade by decade in the major producers.

## §4 — Temporal patterns

Is arthouse share growing? Are recent films received differently?

In [ ]:
# Films per decade — cohort + arthouse share of the universe
def decade(year_col):
    return (year_col // 10 * 10)

cohort_dec = cohort.assign(decade=decade(cohort['releaseYear'])).dropna(subset=['decade'])
universe_dec = universe.assign(decade=decade(universe['releaseYear'])).dropna(subset=['decade'])

decades = list(range(1920, 2030, 10))
c_per_dec = cohort_dec.groupby('decade').size().reindex(decades, fill_value=0)
u_per_dec = universe_dec.groupby('decade').size().reindex(decades, fill_value=0)
share_pct = (c_per_dec / u_per_dec.replace(0, np.nan) * 100).fillna(0)

labels = [f'{int(d)}s' for d in decades]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].bar(labels, c_per_dec.values, color=PRIMARY)
axes[0].set_title('Cohort films per decade')
axes[0].set_ylabel('films')

axes[1].bar(labels, share_pct.values, color=ACCENT)
axes[1].axhline(len(cohort)/len(universe)*100, color=GRAY, ls='--', lw=1,
                label=f'overall ({len(cohort)/len(universe)*100:.1f}%)')
axes[1].set_title('Arthouse share of all films per decade (%)')
axes[1].set_ylabel('%')
axes[1].legend(frameon=False, fontsize=9)

for ax in axes:
    for s in ('top','right'): ax.spines[s].set_visible(False)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout(); plt.show()

# 2020s decade is partial — flag it
n_2020s = int(c_per_dec.loc[2020])
print(f'Note: 2020s contains {n_2020s:,} films but the decade is incomplete (data ends in {int(cohort["releaseYear"].max())}).')

In [ ]:
# Rating + polarization by decade — only on films with usable distributions
coh_pol_dec = coh_pol.assign(decade=decade(coh_pol['releaseYear']).astype('Int64'))
agg = coh_pol_dec.groupby('decade').agg(
    n=('titleId', 'count'),
    mean_rating=('imdbRating', 'mean'),
    mean_polar=('polar', 'mean'),
).round(3)
agg = agg[agg['n'] >= 20]   # require >=20 films/decade for stability
display(Markdown('**Cohort: rating + polarization by decade (decades with >=20 usable films)**'))
display(agg)

**Implication for Publikum (forward-looking strategy):** if arthouse share is rising and recent-decade polarization is up, that's a signal the modern arthouse audience is more divided than past generations — relevant for marketing and platform choice.

## §5 — Genre and thematic structure

The cohort is not one audience. This section names the natural subsegments using IMDb keywords (replacing the dead-on-coverage MovieLens-tag plan).

In [ ]:
def explode_field(s, sep=','):
    return s.dropna().str.split(sep).explode().str.strip()

# Top 10 genres
g = explode_field(cohort['genres'])
top_g = g.value_counts().head(10).to_frame('films')
top_g['pct of cohort'] = (top_g['films'] / len(cohort) * 100).round(1)
display(Markdown('**Top 10 genres in cohort**'))
display(top_g)

# Genre-hybrid share — arthouse with non-traditional genre tags
hybrid_genres = {'Horror', 'Thriller', 'Sci-Fi', 'Action'}
def has_hybrid(s):
    if pd.isna(s): return False
    return bool({p.strip() for p in s.split(',')} & hybrid_genres)
hyb = cohort['genres'].apply(has_hybrid)
print(f'Genre-hybrid share (Horror/Thriller/Sci-Fi/Action): {hyb.sum():,} films ({hyb.mean()*100:.1f}%)')

# Runtime tails
rt = cohort['runtimeMinutes'].dropna()
print(f'\nRuntime: median {rt.median():.0f} min, n = {len(rt):,}')
print(f'  >150 min (slow-cinema range): {(rt>150).sum():,} ({(rt>150).mean()*100:.1f}%)')
print(f'  < 70 min (mid-length / short): {(rt<70).sum():,} ({(rt<70).mean()*100:.1f}%)')

In [ ]:
# Keyword-based subsegments via TF-IDF + NMF topic modeling.
# IMDb keywords are comma-separated phrases like 'based on novel', 'language barrier'.
# Treat each phrase as a single token to preserve phrase identity.
# NMF (vs KMeans) produces more balanced topics on sparse TF-IDF data — KMeans
# tends to collapse 60%+ of films into one catch-all cluster here.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

kw_df = cohort.dropna(subset=['keywords']).copy()

# Drop two kinds of noise:
#  (1) pure title-pattern keywords — describe the title string, not the film
#  (2) decade / century tags — descriptive of era but drown thematic signal
KW_STOP = {
    'one word title', 'two word title', 'three word title', 'four word title',
    'year in title', 'number in title', 'character name in title',
    'place name in title', 'directors name in title',
    '20th century', '21st century',
    '1900s', '1910s', '1920s', '1930s', '1940s', '1950s',
    '1960s', '1970s', '1980s', '1990s', '2000s', '2010s', '2020s',
}

def kw_tokens(s):
    out = []
    for k in s.split(','):
        k = k.strip()
        if not k or k in KW_STOP: continue
        out.append(k.replace(' ', '_'))
    return out

vec = TfidfVectorizer(analyzer=kw_tokens, min_df=10, max_df=0.4)
X = vec.fit_transform(kw_df['keywords'].values)
print(f'Films with keywords: {len(kw_df):,}')
print(f'Vocabulary size after min_df=10, max_df=0.4: {len(vec.vocabulary_):,} keyword phrases')

K = 6
nmf = NMF(n_components=K, random_state=42, init='nndsvda', max_iter=400)
W = nmf.fit_transform(X)   # (n_films, K) topic loadings
H = nmf.components_        # (K, n_terms) topic-term weights

# Hard-assign each film to its dominant topic — but only if the assignment is
# meaningfully strong. Films with low max-loading get a "general" label, since
# they don't have keyword signal pointing strongly at any one topic.
top_loading = W.max(axis=1)
threshold = np.median(top_loading)
kw_df['cluster'] = np.where(top_loading >= threshold, W.argmax(axis=1), -1)
print(f'Loading threshold (median): {threshold:.4f}')
print(f'  films strongly clustered:  {(kw_df["cluster"]>=0).sum():,}')
print(f'  films labelled general:    {(kw_df["cluster"]==-1).sum():,}')

terms = vec.get_feature_names_out()
top_terms_per_cluster = []
for i in range(K):
    top_idx = H[i].argsort()[-8:][::-1]
    top_terms_per_cluster.append([terms[j].replace('_', ' ') for j in top_idx])

print('\nTop 8 keywords per topic:')
for i, terms_i in enumerate(top_terms_per_cluster):
    print(f'  topic {i}: {", ".join(terms_i)}')

In [ ]:
# Cluster summary: size, top countries, mean rating, polarization, sample film.
# Includes the "general / low signal" bucket so the table sums to all films-with-keywords.
coh_pol_idx = coh_pol.set_index('titleId')[['polar', 'mean_from_dist']]
kw_df = kw_df.merge(coh_pol_idx, left_on='titleId', right_index=True, how='left')

def summarize_cluster(sub, label, keywords):
    pol_sub = sub.dropna(subset=['polar'])
    big = sub[sub['totalVotes'].fillna(0) >= 1000].sort_values('totalVotes', ascending=False).head(1)
    if len(big):
        sample_title = big['englishTitle'].fillna(big['originalTitle']).iloc[0]
    else:
        any_film = sub.sort_values('totalVotes', ascending=False, na_position='last').head(1)
        sample_title = any_film['englishTitle'].fillna(any_film['originalTitle']).iloc[0] if len(any_film) else '—'
    return {
        'n':             len(sub),
        'top keywords':  keywords,
        'top countries': ', '.join(sub['mainCountry'].value_counts().head(3).index.tolist()),
        'mean rating':   round(sub['imdbRating'].mean(), 2),
        'polarization':  round(pol_sub['polar'].mean(), 3) if len(pol_sub) else np.nan,
        'sample film':   str(sample_title)[:50],
    }

rows = []
for c in range(K):
    sub = kw_df[kw_df['cluster'] == c]
    rows.append(summarize_cluster(sub, c, ', '.join(top_terms_per_cluster[c][:5])))

# Add a row for the unclustered ("general") bucket
gen_sub = kw_df[kw_df['cluster'] == -1]
rows.append(summarize_cluster(gen_sub, 'general', 'no strong topical signal'))

cluster_summary = pd.DataFrame(rows)
cluster_summary.index = list(range(K)) + ['general']
cluster_summary.index.name = 'cluster'
display(Markdown('**Keyword-based subsegments (NMF, k=6, films below median topic-loading bucketed as "general")**'))
display(cluster_summary)

**Implication for Publikum (target audiences + comparable titles):** these clusters are the closest thing the data supports to named arthouse audience segments. Two films in the same cluster are stronger comps than two films just sharing the arthouse label. Cluster-level rating + polarization tells you which segments are critically beloved vs which divide audiences.

**Caveat on the clusters:** IMDb keywords are noisy free-text. One cluster typically comes through as "explicit content" — that's not a thematic segment, it surfaces a known LLM false-positive mode (erotic/exploitation films misclassified as arthouse). Treat it as a noise pocket, not a real subsegment.

## §6 — Two arthouse modes (rule vs LLM)

The rule-based filter and the LLM scorer catch different kinds of arthouse (`deciding-method.md`). Are `rule_only`, `llm_only`, and `both` three populations or one?

In [ ]:
def summarize_mode(group_name):
    g = cohort[cohort['arthouse_source'] == group_name]
    g_pol = polarization(g)
    rt = g['runtimeMinutes'].dropna()
    lang_g = g['original_language'].fillna(g['firstLanguage'])
    return {
        'n':                len(g),
        'mean rating':      round(g_pol['imdbRating'].mean(), 2) if len(g_pol) else np.nan,
        'polarization':     round(g_pol['polar'].mean(), 3) if len(g_pol) else np.nan,
        'median runtime':   int(rt.median()) if len(rt) else np.nan,
        'top country':      g['mainCountry'].mode().iloc[0] if g['mainCountry'].notna().any() else '—',
        'top genre':        explode_field(g['genres']).value_counts().head(1).index[0] if g['genres'].notna().any() else '—',
        'english share %':  round((lang_g == 'en').mean() * 100, 1),
    }

modes = pd.DataFrame([summarize_mode(s) for s in ['rule_only', 'llm_only', 'both']],
                     index=['rule_only', 'llm_only', 'both'])
display(Markdown('**Comparison across the three arthouse-source groups**'))
display(modes)

# Director overlap
def director_set(group_name):
    return set(
        cohort[cohort['arthouse_source'] == group_name]['directors']
            .dropna().str.split(',').explode().str.strip()
    )

r_dirs = director_set('rule_only')
l_dirs = director_set('llm_only')
b_dirs = director_set('both')
print(f'\nDirectors caught only by rule:        {len(r_dirs):,}')
print(f'Directors caught only by LLM (>=8):   {len(l_dirs):,}')
print(f'Directors in the both group:          {len(b_dirs):,}')
print(f'Rule-only ∩ LLM-only directors:       {len(r_dirs & l_dirs):,}'
      ' (filmmakers where one film hit each method but never the same film)')

**Implication for Publikum (comparable titles):** the three groups are not interchangeable. `rule_only` films come through with festival/distributor signals; `llm_only` films come through name recognition. When picking comps for a new project, match on the *mode* — pick rule-only comps for an industry-arthouse positioning brief, llm-only comps for an auteur-driven brief.

## §7 — Commercial signals (small-n caveat)

Only ~5% of the cohort has both budget and revenue. Treat everything below as **directional**, not statistical.

In [ ]:
# Filter: both budget and revenue >0, and budget >= $50k (sub-$50k often a metadata error)
roi_df = cohort[(cohort['budget'].fillna(0) > 0) & (cohort['revenue'].fillna(0) > 0)].copy()
roi_df = roi_df[roi_df['budget'] >= 50_000]
roi_df['roi'] = roi_df['revenue'] / roi_df['budget']

print(f'Films with usable budget+revenue (>=$50k budget): n = {len(roi_df):,}')
print(f'Median ROI:        {roi_df["roi"].median():.2f}x')
print(f'25th–75th pctile:  {roi_df["roi"].quantile(0.25):.2f}x – {roi_df["roi"].quantile(0.75):.2f}x')
print(f'Share of films profitable (ROI > 1): {(roi_df["roi"]>1).mean()*100:.1f}%')

# Top 10 ROI films
top_roi = roi_df.nlargest(10, 'roi')[['englishTitle', 'originalTitle', 'releaseYear',
                                       'mainCountry', 'budget', 'revenue', 'roi', 'imdbRating']].copy()
top_roi['title'] = top_roi['englishTitle'].fillna(top_roi['originalTitle'])
top_roi = top_roi[['title', 'releaseYear', 'mainCountry', 'budget', 'revenue', 'roi', 'imdbRating']]
top_roi['budget'] = top_roi['budget'].apply(lambda x: f'${x:,.0f}')
top_roi['revenue'] = top_roi['revenue'].apply(lambda x: f'${x:,.0f}')
top_roi['roi'] = top_roi['roi'].round(1)
display(Markdown(f'**Top 10 ROI films in cohort (n={len(roi_df):,}, directional only)**'))
display(top_roi.reset_index(drop=True))

In [ ]:
# Budget vs rating scatter — does spending more on arthouse correlate with reception?
fig, ax = plt.subplots(figsize=(8, 4.5))
sub = roi_df.dropna(subset=['imdbRating'])
ax.scatter(sub['budget'], sub['imdbRating'], alpha=0.55, color=PRIMARY, s=22)
ax.set_xscale('log')
ax.set_xlabel('Budget (USD, log scale)')
ax.set_ylabel('IMDb rating')
ax.set_title(f'Budget vs IMDb rating (n={len(sub):,}, ~5% of cohort — interpret cautiously)')
for s in ('top','right'): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

# Spearman correlation since the relationship may be non-linear
from scipy.stats import spearmanr
rho, p = spearmanr(sub['budget'], sub['imdbRating'])
print(f'Spearman correlation budget vs rating: rho = {rho:.3f} (p = {p:.3f})')

**Implication for Publikum (commercial):** at the cohort level, ROI is highly skewed — a small number of crossover hits dominate. Useful as a list of comparable success stories (the top-10 table), not as a budget-prediction model.

## Implications, mapped to Publikum's decision questions

| Decision question | Sections that speak to it | Concrete deliverable |
|---|---|---|
| **Project positioning** | §2 polarization, §5 hybrid genres, §6 arthouse modes | A new film gets a *predicted polarization band* + a *mode classification* (industry-arthouse vs auteur-arthouse) |
| **Target audiences & segments** | §5 keyword clusters, §2 polarization | Six named subsegments with rating + polarization profiles |
| **Market & country strategy** | §3 country counts + per-capita, §4 country×decade heatmap | Per-capita ranking flags small-but-mighty markets that outweigh raw-volume leaders |
| **Comparable-title analysis** | §5 cluster, §6 mode, §2 polarization shape | A comp shortlist for a new film should match on *cluster + mode + polarization band* — not just on the arthouse label alone |

## Open questions / not-yet-answered

- **Cinephile-discovered films** would need MovieLens coverage (currently 1.5%) — can't do until that improves.
- **Commercial regression / pricing model** would need budget+revenue coverage past ~25%.
- **Plot-text embeddings** for thematic geography — viable on the 73% with plot summaries, parked as future work.